In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")
import random
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# 変数、乱数の設定

In [ ]:
class CFG:
    VER = 1
    AUTHOR = "suba"
    COMPETITION = "atmacup17"
    DATA_PATH = Path("/workspace/kaggle_llm_book/ch3/data/takaito/atmacup17")
    MODEL_PATH = "/workspace/kaggle_llm_book/ch3/model/suba_deberta-v3-large_seed42_ver1-fold0/checkpoint-140"
    MODEL_NAME = "deberta-v3-large"
    MAX_LENGTH = 2048
    EPOCHS = 3
    STEPS = 20
    WARMUP_RATIO = 0.05
    TRAIN_BATCH_SPLIT = 8
    TRAIN_BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 64
    LEARNING_RATE = 2e-5
    OPTIM = "adamw_torch"
    USE_GPU = torch.cuda.is_available()
    SEED = 42
    N_SPLIT = 2
    TARGET_COL = "Recommended IND"
    TARGET_COL_CLASS_NUM = 2
    METRIC = "auc"
    METRIC_MAXIMIZE_FLAG = True
    PREFIX  = f"{AUTHOR}_{MODEL_NAME}_seed{SEED}_ver{VER}"
    OUTPUT_DIR = "./model"
    MODEL_PATH_DICT = {
        0: "/workspace/kaggle_llm_book/ch3/model/suba_deberta-v3-large_seed42_ver1-fold0/checkpoint-140",
        1: "/workspace/kaggle_llm_book/ch3/model/suba_deberta-v3-large_seed42_ver1-fold1/checkpoint-200"
    }

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if CFG.USE_GPU:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything(CFG.SEED)

# データの読み込み、前処理

In [ ]:
clothing_master_df = pd.read_csv(CFG.DATA_PATH / "clothing_master.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")

In [ ]:
def make_prompt_column(df):
    df["prompt"] = df["Title"] + " " + df["Review Text"]
    return df

def preprocessing(df, clothing_master_df):
    df["Title"] = df["Title"].fillna("")
    df["Review Text"] = df["Review Text"].fillna("")
    df = df.merge(clothing_master_df, on="Clothing ID", how="left")
    df = make_prompt_column(df)
    return df

In [ ]:
test_df = preprocessing(test_df, clothing_master_df)

In [ ]:
display(test_df)

In [ ]:
print(test_df[["prompt"]].iloc[0].values)

# トークナイザの設定

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_PATH)
def tokenize(sample, tokenizer):
    return tokenizer(sample["prompt"], max_length=CFG.MAX_LENGTH, truncation=True)

# データセットの作成

In [ ]:
def mk_dataset(df, tokenizer):
    raw_datasets = Dataset.from_pandas(df)
    tokenized_datasets = raw_datasets.map(
        tokenize, 
        fn_kwargs={"tokenizer": tokenizer}
    ).remove_columns(["prompt"])
    return tokenized_datasets

In [ ]:
ds_test = mk_dataset(test_df[["prompt"]], tokenizer)

# 推論

In [ ]:
def get_args(CFG, fold):
    args = TrainingArguments(
        output_dir=f"{CFG.OUTPUT_DIR}/{CFG.PREFIX}-fold{fold}",
        report_to="none",
        eval_strategy="steps",
        logging_strategy="steps",
        save_strategy="steps",
        eval_steps=CFG.STEPS,
        logging_steps=CFG.STEPS,
        save_steps=CFG.STEPS,
        save_total_limit=1,
        metric_for_best_model=CFG.METRIC,
        greater_is_better=CFG.METRIC_MAXIMIZE_FLAG,
        optim=CFG.OPTIM,
        learning_rate=CFG.LEARNING_RATE,
        lr_scheduler_type="linear",
        warmup_ratio=CFG.WARMUP_RATIO,
        per_device_train_batch_size=CFG.TRAIN_BATCH_SIZE // CFG.TRAIN_BATCH_SPLIT,
        per_device_eval_batch_size=CFG.EVAL_BATCH_SIZE,
        gradient_accumulation_steps=CFG.TRAIN_BATCH_SPLIT,
        num_train_epochs=CFG.EPOCHS,
        fp16=True,
        seed=CFG.SEED,
        data_seed=CFG.SEED,
    )
    return args

In [ ]:
test_predictions = np.zeros(len(test_df))
for fold in range(CFG.N_SPLIT):
    model_path = CFG.MODEL_PATH_DICT[fold]
    # モデルの読み込み & 設定
    config = AutoConfig.from_pretrained(CFG.MODEL_PATH)
    config.num_labels = CFG.TARGET_COL_CLASS_NUM
    model = AutoModelForSequenceClassification.from_pretrained(model_path, config=config)
    trainer = Trainer(
        model=model,
        args=get_args(CFG, fold),
        train_dataset=ds_test,
        eval_dataset=ds_test,
        data_collator=DataCollatorWithPadding(tokenizer),
        tokenizer=tokenizer,
    )

    # テストセットの推論
    test_preds = trainer.predict(ds_test).predictions
    test_predictions += torch.softmax(torch.tensor(test_preds), dim=1).numpy()[:, 1] / CFG.N_SPLIT

In [ ]:
test_df["target"] = test_predictions
test_df[["target"]].to_csv(f"{CFG.PREFIX}_submission.csv", index=False)

In [ ]:
test_df